# Data Loding + Features

In [ ]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def load_dataset(folder):
    all_files = glob.glob(f"{folder}/**/*.csv", recursive=True)
    samples = []

    print(f"Found {len(all_files)} files in '{folder}'")

    for i, filepath in enumerate(all_files, start=1):

        if i % 500 == 0 or i == 1:
            print(f"[{folder}] Processed {i}/{len(all_files)} files")

        df = pd.read_csv(filepath)

        file_id = df['file_id'].iloc[0]
        label = df['label'].iloc[0] if 'label' in df.columns else None

        features = {}

        for col in ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']:
            features[f'{col}_mean'] = df[col].mean()
            features[f'{col}_std'] = df[col].std()
            features[f'{col}_min'] = df[col].min()
            features[f'{col}_max'] = df[col].max()
            features[f'{col}_median'] = df[col].median()

            features[f'{col}_range'] = df[col].max() - df[col].min()
            features[f'{col}_q25'] = df[col].quantile(0.25)
            features[f'{col}_q75'] = df[col].quantile(0.75)

        # Magnitude
        mag = np.sqrt(
            df['mean_x']**2 +
            df['mean_y']**2 +
            df['mean_z']**2
        )

        features['mag_mean'] = mag.mean()
        features['mag_std'] = mag.std()
        features['mag_min'] = mag.min()
        features['mag_max'] = mag.max()

        # Targeted features for class 1 vs class 2
        features['mean_y_iqr'] = df['mean_y'].quantile(0.75) - df['mean_y'].quantile(0.25)
        features['mean_y_q10'] = df['mean_y'].quantile(0.10)
        features['mean_y_q90'] = df['mean_y'].quantile(0.90)

        features['mean_x_iqr'] = df['mean_x'].quantile(0.75) - df['mean_x'].quantile(0.25)
        features['mean_x_q10'] = df['mean_x'].quantile(0.10)
        features['mean_x_q90'] = df['mean_x'].quantile(0.90)

        features['mag_iqr'] = mag.quantile(0.75) - mag.quantile(0.25)
        features['mag_q10'] = mag.quantile(0.10)
        features['mag_q90'] = mag.quantile(0.90)

        features['file_id'] = file_id
        features['label'] = label

        samples.append(features)

    print(f"Finished loading '{folder}'")
    return pd.DataFrame(samples)


train_df = load_dataset("train")
test_df = load_dataset("test")

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")

# Prepare x,y

In [ ]:
feature_cols = [c for c in train_df.columns
                if c not in ['file_id', 'label']]

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test = test_df[feature_cols].values

print(f"Number of features: {len(feature_cols)}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")


# TRAIN / VALIDATION SPLIT


X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)


# Sample weights

In [ ]:
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_tr
)

# XGBoost model

In [10]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

# Train model
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1
)

xgb.fit(
    X_tr,
    y_tr,
    sample_weight=sample_weights
)



y_val_pred = xgb.predict(X_val)

f1 = f1_score(
    y_val,
    y_val_pred,
    average="macro"
)

print(f"Macro F1-score: {f1:.4f}")



xgb.fit(
    X_train,
    y_train
)



test_predictions = xgb.predict(X_test)

submission = pd.DataFrame({
    "Id": test_df["file_id"],
    "Label": test_predictions.astype(int)
})

submission.to_csv(
    "final_submission.csv",
    index=False
)

print("final_submission.csv created")
print(submission.head())

Macro F1-score: 0.7601
final_submission.csv created
      Id  Label
0  11021      0
1  11022      0
2  11023      0
3  11024      0
4  11025      0
